# Task 1.2: Key Assumptions

**Paper:** Exact Discovery of Time Series Motifs  
**Authors:** Abdullah Mueen, Eamonn Keogh, Qiang Zhu, Sydney Cash, Brandon Westover  
**Venue:** KDD 2009 (ACM SIGKDD)  
**Student:** Abhishek (m23csai230137)

---

## Assumption 1

**Assumption:** The Euclidean distance is a valid and sufficient similarity measure for identifying meaningful motifs in time series data.

**Why the method needs it:** The entire MK algorithm — its triangle inequality pruning, its early abandoning optimisation, and its lower bound computation — depends on the distance measure being a **metric** (satisfying the triangle inequality). The paper states: *"our method is valid for any distance measure that is a metric. We use Euclidean distance"* (Section 2). If the distance function were not a metric, the lower bounds `|d(ref, Dᵢ) - d(ref, Dⱼ)| ≤ d(Dᵢ, Dⱼ)` would be invalid, and the algorithm could prune the true motif pair — producing an incorrect result. Furthermore, the paper's argument that DTW is unnecessary for large datasets (Section 4.3.2) is based on the empirical observation that Euclidean distance and DTW converge for close pairs, which only holds when the motif distance is very small.

**Violation scenario:** Consider time series data where the same underlying pattern appears with **temporal warping** — for example, heart rate signals where the same cardiac event (say a premature ventricular contraction) occurs at different speeds in different patients. Two instances of the same motif might be temporally stretched or compressed relative to each other. Under Euclidean distance, these would appear far apart (high point-wise error), and the algorithm would fail to identify them as the true motif. A warping-aware measure like DTW would correctly identify them as similar.

**Paper reference:** Section 2 (distance measure definition), Section 4.3.2 ("Why not use DTW or Uniform Scaling?"), Figure 11 (scatter plot showing Euclidean vs DTW divergence for distant pairs).

## Assumption 2

**Assumption:** Randomly chosen reference time series from the database produce a sufficient spread (high standard deviation in distances) to generate tight lower bounds for pruning.

**Why the method needs it:** The MK algorithm's speedup over brute force depends entirely on how many candidate pairs it can prune using triangle inequality lower bounds. The tightness of these lower bounds depends on the **position** of the reference point relative to the data distribution. The algorithm selects the reference with the highest standard deviation in its distances to other time series (Table 3, Lines 8–9), because high variance implies that the 1D projection will have a wide spread, making it easier to distinguish objects that are far apart. The paper acknowledges: *"Not all objects are equally good as a reference point"* (Section 3.1). If all references happen to be at the centroid of the data (equidistant from everything), all lower bounds would be near zero and no pruning would occur, degrading the algorithm to brute force.

**Violation scenario:** Consider a database of time series that all lie approximately on a hypersphere — for example, Z-normalised white noise of constant length. Every time series would be roughly the same distance from every other, so any randomly chosen reference would have **near-zero standard deviation** in its distance vector. The lower bounds would all be approximately zero, no pairs would be pruned, and the algorithm would have to compute true distances for all O(m²) pairs, losing its speed advantage entirely.

**Paper reference:** Table 3, Lines 8–9 (reference selection by standard deviation), Section 3.1 ("Not all objects are equally good as a reference point"), Section 3.2.1 (worst case: *"the motif has distance larger than any lower bound computed using a random reference"*), Figure 8 (effect of number of reference points).

## Assumption 3

**Assumption:** The true motif pair has a distance significantly smaller than the average pairwise distance in the database, so that the best-so-far converges to a small value early in the search.

**Why the method needs it:** The algorithm's practical speed depends on two linked mechanisms: (1) early abandoning is effective only when the best-so-far is small relative to most pairwise distances (Section 4.3.1), and (2) the offset-based search terminates early only when the best-so-far is small enough that most lower bounds exceed it. The paper explicitly connects this to the **birthday paradox** analogy (Section 4.3.1, Figure 9): in a large database, the motif distance drops dramatically with database size because there are m(m−1)/2 possible pairs. If the motif distance is not significantly below the average distance — for instance, in a very small database — neither early abandoning nor the pruning strategy would be effective, and the algorithm would approach its worst-case O(m²n) complexity.

**Violation scenario:** A very small database (say m = 10 short time series) of unrelated random sequences. With only 45 possible pairs, the birthday paradox effect is weak: no pair is expected to be dramatically closer than average. The best-so-far would remain relatively large throughout the search, early abandoning would rarely trigger, and the lower bound pruning from reference points would fail to eliminate many pairs. In this scenario, the MK algorithm would have **no meaningful speedup** over brute force and would actually be slower due to the overhead of computing reference distances and sorting.

**Paper reference:** Section 4.3.1 ("Why is Early-Abandoning so Effective?"), Figure 9 (motif distance vs. average distance vs. dataset size), Section 4.1 (random walk as the hardest case because *"we should not expect a very close motif pair to exist"*).